In [9]:
import pandas as pd

df = pd.read_excel("../raw/E Commerce Dataset.xlsx", sheet_name=1)
df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   str    
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   str    
 7   Gender                       5630 non-null   str    
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   str    
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   str    
 13  NumberOfAddress              

In [11]:
df.isnull().sum()

CustomerID                       0
Churn                            0
Tenure                         264
PreferredLoginDevice             0
CityTier                         0
WarehouseToHome                251
PreferredPaymentMode             0
Gender                           0
HourSpendOnApp                 255
NumberOfDeviceRegistered         0
PreferedOrderCat                 0
SatisfactionScore                0
MaritalStatus                    0
NumberOfAddress                  0
Complain                         0
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
CashbackAmount                   0
dtype: int64

In [20]:
import pickle
from pathlib import Path

# 1) Basic cleanup
TARGET_COL = "Churn"
df_clean = df.copy()
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
rows_after = len(df_clean)
print(f"Rows before/after duplicate removal: {rows_before} -> {rows_after}")

# 2) Full-dataset imputation (cleaning-only flow, no train/valid split)
feature_cols = [col for col in df_clean.columns if col != TARGET_COL]
num_cols = df_clean[feature_cols].select_dtypes(include=["number"]).columns.tolist()
cat_cols = df_clean[feature_cols].select_dtypes(exclude=["number"]).columns.tolist()
missing_cols = [col for col in feature_cols if df_clean[col].isna().any()]

num_fill_values = df_clean[num_cols].median().to_dict() if num_cols else {}
cat_fill_values = {}
for col in cat_cols:
    mode_values = df_clean[col].mode(dropna=True)
    cat_fill_values[col] = mode_values.iloc[0] if not mode_values.empty else "Unknown"

# 3) Apply rules to full dataset
X_all = df_clean[feature_cols].copy()
if num_cols:
    X_all[num_cols] = X_all[num_cols].fillna(value=num_fill_values)
if cat_cols:
    X_all[cat_cols] = X_all[cat_cols].fillna(value=cat_fill_values)

if TARGET_COL in df_clean.columns:
    df_imputed = pd.concat([X_all, df_clean[TARGET_COL]], axis=1)
else:
    df_imputed = X_all.copy()

# Ensure original column order
df_imputed = df_imputed[df_clean.columns.tolist()]

print("Total remaining nulls in df_imputed:", int(df_imputed.isnull().sum().sum()))
print("Columns with nulls handled:", len(missing_cols))
display(df_imputed.isnull().sum().sort_values(ascending=False).head(20))

# 4) Save reusable cleaning/imputation rules
artifact_path = Path("imputation_rules.pkl")
with open(artifact_path, "wb") as file:
    pickle.dump(
        {
            "cleaning_mode": "full_dataset",
            "target_col": TARGET_COL,
            "column_order": df_clean.columns.tolist(),
            "num_cols": num_cols,
            "cat_cols": cat_cols,
            "missing_cols": missing_cols,
            "num_fill_values": num_fill_values,
            "cat_fill_values": cat_fill_values,
        },
        file,
    )

print(f"Saved imputation rules to: {artifact_path.resolve()}")

Rows before/after duplicate removal: 5630 -> 5630
Total remaining nulls in df_imputed: 0
Columns with nulls handled: 7


CustomerID                     0
Churn                          0
Tenure                         0
PreferredLoginDevice           0
CityTier                       0
WarehouseToHome                0
PreferredPaymentMode           0
Gender                         0
HourSpendOnApp                 0
NumberOfDeviceRegistered       0
PreferedOrderCat               0
SatisfactionScore              0
MaritalStatus                  0
NumberOfAddress                0
Complain                       0
OrderAmountHikeFromlastYear    0
CouponUsed                     0
OrderCount                     0
DaySinceLastOrder              0
CashbackAmount                 0
dtype: int64

Saved imputation rules to: E:\KENEX_ETL\data\preprocessed\imputation_rules.pkl


In [21]:
# 10) Export clean preprocessed CSV with same columns as raw data
from pathlib import Path

raw_columns = df.columns.tolist()
df_export = df_imputed[raw_columns].copy()

preferred_output_csv_path = Path("churn_data_preprocessed.csv")
fallback_output_csv_path = Path("churn_data_preprocessed_same_columns.csv")

try:
    df_export.to_csv(preferred_output_csv_path, index=False)
    final_output_csv_path = preferred_output_csv_path
except PermissionError:
    df_export.to_csv(fallback_output_csv_path, index=False)
    final_output_csv_path = fallback_output_csv_path

comparison = pd.DataFrame(
    {
        "metric": ["rows", "columns", "total_null_values", "duplicate_rows"],
        "raw_data": [
            int(df.shape[0]),
            int(df.shape[1]),
            int(df.isnull().sum().sum()),
            int(df.duplicated().sum()),
        ],
        "preprocessed_data": [
            int(df_export.shape[0]),
            int(df_export.shape[1]),
            int(df_export.isnull().sum().sum()),
            int(df_export.duplicated().sum()),
        ],
    }
)

print(f"Saved clean CSV to: {final_output_csv_path.resolve()}")
print("Column match with raw:", raw_columns == df_export.columns.tolist())
print("Exported columns:", len(df_export.columns))
display(comparison)

Saved clean CSV to: E:\KENEX_ETL\data\preprocessed\churn_data_preprocessed.csv
Column match with raw: True
Exported columns: 20


,metric,raw_data,preprocessed_data
0,rows,5630,5630
1,columns,20,20
2,total_null_values,1856,0
3,duplicate_rows,0,0
